# Evaluación baseline: falla de prompting directo

Este notebook mide, con evidencia propia, qué tan seguido falla un modelo abierto
de hasta 8B cuando se le pide directamente (sin fine-tuning ni técnicas adicionales)
generar el SQL necesario para responder preguntas de negocio sobre `business.db`.

Corresponde a la evidencia propia que exige el punto "Failure diagnosis" de la
rúbrica del Entregable 1, además de las cifras ya citadas de la literatura.

Requiere GPU (Runtime > Change runtime type > T4 GPU en Colab).

In [ ]:
!pip install -q transformers accelerate bitsandbytes

## 1. Cargar la base de datos y las preguntas

Sube `business.db` y `questions.json` (carpeta `data/` del repositorio) a esta sesión de Colab, o móntalos desde Google Drive.

In [ ]:
import json
import sqlite3
import re
from pathlib import Path

DB_PATH = "business.db"
Q_PATH = "questions.json"

data = json.loads(Path(Q_PATH).read_text(encoding="utf-8"))
SCHEMA = data["schema"]
QUESTIONS = data["questions"]
print(f"{len(QUESTIONS)} preguntas cargadas.")
print(SCHEMA)

## 2. Cargar el modelo candidato

Cambia `MODEL_NAME` para probar los otros candidatos (Qwen2.5-Coder-3B-Instruct, Llama-3.1-8B-Instruct).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
print("Modelo cargado:", MODEL_NAME)

## 3. Prompt de prompting directo (baseline, sin técnicas adicionales)

In [ ]:
PROMPT_TEMPLATE = """Eres un asistente que traduce preguntas de negocio a SQL.

Esquema de la base de datos:
{schema}

Pregunta: {question}

Responde unicamente con la o las consultas SQL necesarias para responder la
pregunta, separadas por punto y coma. No expliques nada, no uses markdown."""


def ask_model(question):
    prompt = PROMPT_TEMPLATE.format(schema=SCHEMA, question=question)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    output = model.generate(
        inputs, max_new_tokens=300, do_sample=False, temperature=None, top_p=None
    )
    text = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)
    return text.strip()

## 4. Extraer y ejecutar el SQL generado

In [ ]:
def extract_sql_statements(text):
    text = re.sub(r"```sql|```", "", text, flags=re.IGNORECASE)
    parts = [p.strip() for p in text.split(";")]
    return [p for p in parts if p and p.upper().startswith("SELECT")]


def run_sql(conn, sql):
    try:
        cur = conn.cursor()
        cur.execute(sql)
        return cur.fetchall()
    except Exception as e:
        return f"ERROR: {e}"


def matches_gold(generated_results, gold_result):
    gold_set = [set(map(str, r)) for r in gold_result]
    gen_set = [set(map(str, r)) for r in generated_results if isinstance(r, list)]
    return all(any(g == gset for gset in gen_set) for g in gold_set)

## 5. Correr el baseline sobre las 15 preguntas

In [ ]:
conn = sqlite3.connect(DB_PATH)
results = []

for q in QUESTIONS:
    raw_output = ask_model(q["question"])
    statements = extract_sql_statements(raw_output)
    executed = [run_sql(conn, s) for s in statements]
    correct = matches_gold(executed, q["gold_result"])

    results.append({
        "id": q["id"],
        "type": q["type"],
        "question": q["question"],
        "model_output": raw_output,
        "n_statements_generated": len(statements),
        "correct": correct,
    })
    print(f"{q['id']} [{q['type']}] correcto={correct}")

conn.close()

## 6. Reportar exactitud (esta es la evidencia propia para el documento)

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
overall = df["correct"].mean()
by_type = df.groupby("type")["correct"].mean()

print(f"Exactitud de ejecucion global: {overall:.1%}")
print("\nPor tipo de pregunta:")
print(by_type)

df.to_json("baseline_results.json", orient="records", indent=2, force_ascii=False)
print("\nGuardado en baseline_results.json")

## 7. Siguientes pasos

- Repetir con `Qwen2.5-Coder-3B-Instruct` y `Llama-3.1-8B-Instruct` para comparar los tres candidatos.
- Guardar `baseline_results.json` en `results/` y subirlo al repositorio.
- Estos números, junto a las cifras citadas de la literatura (BIRD, JudgeSQL, TinyLLM), son la evidencia que respalda la sección "Diagnóstico de la falla" del documento.